# Train realization models (`t5-small`)

You are **not** training a translator. Fine-tune `t5-small` so it turns a **semantic frame** into a sentence. Latin Dhivehi and English only — no Thaana goes into either model.

```text
SUBJECT=I | ACTION=go | LOCATION=Male | TENSE=future | POLARITY=affirmative | REGISTER=spoken
        ↓
I will go to Male.
```

```text
SUBJECT=aharen | ACTION=dhaa | LOCATION=male | TENSE=future | POLARITY=affirmative | REGISTER=spoken
        ↓
aharen maleah dhaanan
```

| Run | Files | Output | Used for |
|---|---|---|---|
| English | `en_train.jsonl`, `en_valid.jsonl` | `en_realize` | Dhivehi → English |
| Dhivehi Latin | `dv_train.jsonl`, `dv_valid.jsonl` | `dv_realize` | English → Dhivehi Latin |

This notebook trains **both** in one session. Enable a **T4 GPU** first: **Runtime → Change runtime type → T4 GPU**.

Keep this tab open and visible. Idle Colab sessions die after about 90 minutes.

## Training data

Upload the four JSONL files from `latin-mv-tlt/data/realize/` on your laptop. Each line is `{input, target, direction}` — a frame string to a sentence. There is no `grammar:` prefix. If a line looks like `grammar: I Male go future`, you uploaded the old APE pairs; stop and use `data/realize/` instead.

Current sizes (measured, `data/realize/stats.json`):

| File | Lines |
|---|---|
| `en_train.jsonl` | 14526 |
| `en_valid.jsonl` | 1615 |
| `dv_train.jsonl` | 12843 |
| `dv_valid.jsonl` | 1427 |

The corpus is built from a curated slot vocabulary of about sixty content words (7 subjects, 16 verbs, 12 objects, 5 locations, 7 time words). A checkpoint trained on it realizes those slots well and generalises no further than they reach.

`REGISTER=written` vs `REGISTER=spoken` is a frame slot: written Dhivehi clauses end in `eve`. Location goals use the dative (`maleah`, not a bare `Male`). Slot values are plain ASCII.

Do not regenerate the JSONL after you have already uploaded it, or the Colab copy no longer matches disk.

## 1 — Mount Google Drive

Authorize when Colab prompts. Checkpoints are copied to `/content/drive/MyDrive/` after training so a dropped session does not lose the zips.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2 — GPU check

You must see `GPU available: True`. If not: **Runtime → Change runtime type → T4 GPU**, then re-run from the Drive cell.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Runtime → Change runtime type → T4 GPU, then re-run.")

## 3 — Install libraries

`transformers>=4.46` is required because the Trainer uses `processing_class=`. Red dependency-conflict lines about unrelated Colab packages are noise.

In [ ]:
%pip install -q -U "transformers>=4.46" "accelerate>=1.1" sentencepiece
print("done")

## 4 — Upload the four JSONL files

Ctrl+click all four from `latin-mv-tlt\data\realize\`:

- `en_train.jsonl`
- `en_valid.jsonl`
- `dv_train.jsonl`
- `dv_valid.jsonl`

In [ ]:
from google.colab import files
print("Select ALL FOUR: en_train.jsonl, en_valid.jsonl, dv_train.jsonl, dv_valid.jsonl (Ctrl+click)")
uploaded = files.upload()
print(sorted(uploaded.keys()))

## 5 — Verify files

Fails if a file is missing, the line count does not match `stats.json`, or a line still starts with `grammar:`.

In [ ]:
import json
from pathlib import Path

EXPECTED = {
    "en_train.jsonl": 14526,
    "en_valid.jsonl": 1615,
    "dv_train.jsonl": 12843,
    "dv_valid.jsonl": 1427,
}

for name, n in EXPECTED.items():
    path = Path(name)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name}. Upload the four files from latin-mv-tlt/data/realize/"
        )
    lines = [ln for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    print(f"{name}: {len(lines)} (expected {n})")
    if len(lines) != n:
        raise ValueError(f"{name} has {len(lines)} lines, expected {n}")
    row = json.loads(lines[0])
    sample = row["input"]
    if sample.startswith("grammar:"):
        raise ValueError("Old APE pairs. Use data/realize/, not dhivehi-latin-slm/data/ape/")
    if not sample.startswith("SUBJECT="):
        raise ValueError(f"{name} input is not a frame string: {sample[:80]}")
    print("  sample input:", sample)
    print("  sample target:", row["target"])
print("ok")

## 6 — Shared trainer

Same as `tools/train_t5_realize.py`, with Colab batch 16 and fp16. If Colab says `CUDA out of memory`, set `BATCH = 8` and re-run from this cell.

`SMOKE = True` is a 30-second wiring check only. Do not download or evaluate a smoke checkpoint as if it were the real model.

In [ ]:
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, TrainingArguments

MODEL = "t5-small"
MAX_LEN = 128
EPOCHS = 3
BATCH = 16  # set to 8 if CUDA out of memory
SMOKE = False  # True = 8 train / 2 valid / 1 epoch, wiring only


class FrameDataset(Dataset):
    def __init__(self, path, tokenizer, max_len=MAX_LEN):
        self.rows = [
            json.loads(line)
            for line in Path(path).read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        row = self.rows[idx]
        source = self.tokenizer(
            row["input"], truncation=True, padding="max_length", max_length=self.max_len
        )
        target = self.tokenizer(
            row["target"], truncation=True, padding="max_length", max_length=self.max_len
        )
        labels = [
            token if token != self.tokenizer.pad_token_id else -100
            for token in target["input_ids"]
        ]
        return {
            "input_ids": torch.tensor(source["input_ids"]),
            "attention_mask": torch.tensor(source["attention_mask"]),
            "labels": torch.tensor(labels),
        }


def train_realize(train_path, valid_path, out_dir):
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
    train_ds = FrameDataset(train_path, tokenizer)
    valid_ds = FrameDataset(valid_path, tokenizer)
    epochs = EPOCHS
    if SMOKE:
        train_ds.rows = train_ds.rows[:8]
        valid_ds.rows = valid_ds.rows[:2]
        epochs = 1
    print("Train:", len(train_ds), "Valid:", len(valid_ds))
    print("Sample input:", train_ds.rows[0]["input"])
    print("Sample target:", train_ds.rows[0]["target"])
    args = TrainingArguments(
        output_dir=f"{out_dir}/runs",
        num_train_epochs=epochs,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        fp16=torch.cuda.is_available(),
        report_to=[],
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        processing_class=tokenizer,
    )
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    print("Saved to", out_dir)
    return trainer, model, tokenizer


def probe(model, tokenizer, text):
    model.eval()
    ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=64)
    print(tokenizer.decode(out[0], skip_special_tokens=True))

## 7 — Train English realization

Both training loss and validation loss should fall. Screenshot the table for Chapter 5.

- Validation loss rising while training loss falls → overfitting. Try 2 epochs next time.
- Loss stuck near 0 from the first step → the model is copying. Check that `input` and `target` are different.

Record what you actually ran: `t5-small`, 14526 / 1615 pairs, 3 epochs, lr `5e-5`, batch 16, fp16, T4. If you change any of those, record the values you used.

In [ ]:
trainer, model, tokenizer = train_realize("en_train.jsonl", "en_valid.jsonl", "en_realize")

## 8 — English sanity probe

A good sign: something like `I will go to Male.`

A bad sign: the model prints the frame back unchanged, or empty text.

In [ ]:
probe(
    model,
    tokenizer,
    "SUBJECT=I | ACTION=go | LOCATION=Male | TENSE=future | POLARITY=affirmative",
)

## 9 — Train Dhivehi Latin realization

Frees the English model from GPU memory first so the second run does not OOM.

In [ ]:
import gc

del trainer, model, tokenizer
gc.collect()
torch.cuda.empty_cache()

trainer, model, tokenizer = train_realize("dv_train.jsonl", "dv_valid.jsonl", "dv_realize")

## 10 — Dhivehi Latin sanity probe

A good sign: `aharen maleah dhaanan` (or close). Not Thaana. The dative `-ah` on the goal is obligatory.

In [ ]:
probe(
    model,
    tokenizer,
    "SUBJECT=aharen | ACTION=dhaa | LOCATION=male | TENSE=future | POLARITY=affirmative",
)

## 11 — Save checkpoints

Each zip is on the order of 200–250 MB. Drive is the reliable copy (`/content/drive/MyDrive/`). Browser download is a second copy; allow pop-ups for `colab.research.google.com` if it does not start.

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

DRIVE = Path("/content/drive/MyDrive")

for name in ("en_realize", "dv_realize"):
    zip_path = Path(shutil.make_archive(name, "zip", name))
    dest = DRIVE / f"{name}.zip"
    shutil.copy(zip_path, dest)
    print(f"Drive: {dest} ({dest.stat().st_size / 1e6:.0f} MB)")
    files.download(str(zip_path))

## After Colab

1. Extract the zips on the laptop.
2. Move the folders to:

```text
latin-mv-tlt\models\en_realize\
latin-mv-tlt\models\dv_realize\
```

You should see `model.safetensors` (large), `config.json`, and tokenizer files. `models/` is gitignored on purpose. Do not commit 200 MB weights.

A local `models/` folder is for Python checks and later conversion. It does **not** make fluent output appear in the website. `src/core/realization/runner.ts` loads Hugging Face repo IDs through Transformers.js. Until `VITE_EN_REALIZE_MODEL` and `VITE_DV_REALIZE_MODEL` point at real repos:

- Realization = **Not loaded**
- Final translation = **Unavailable**
- Breakdown still shows the frame (that is correct)

Do not invent BLEU/chrF until you measure them on this pipeline. Do not treat a smoke checkpoint as the real model.